In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [51]:
#!pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [52]:
df = pd.read_csv("../dataset/dataset_phishing.csv")

Identifying data set

In [53]:
df.columns

Index(['url', 'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens',
       'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore',
       'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma',
       'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com',
       'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url',
       'ratio_digits_host', 'punycode', 'port', 'tld_in_path',
       'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains',
       'prefix_suffix', 'random_domain', 'shortening_service',
       'path_extension', 'nb_redirection', 'nb_external_redirection',
       'length_words_raw', 'char_repeat', 'shortest_words_raw',
       'shortest_word_host', 'shortest_word_path', 'longest_words_raw',
       'longest_word_host', 'longest_word_path', 'avg_words_raw',
       'avg_word_host', 'avg_word_path', 'phish_hints', 'domain_in_brand',
       'brand_in_subdomain', 'brand_in_path', 'suspecious_tld',
       'statistical_report', 

In [54]:
df.dtypes

url                  str
length_url         int64
length_hostname    int64
ip                 int64
nb_dots            int64
                   ...  
web_traffic        int64
dns_record         int64
google_index       int64
page_rank          int64
status               str
Length: 89, dtype: object

In [55]:
df['status'].value_counts()

status
legitimate    5715
phishing      5715
Name: count, dtype: int64

In [56]:
print("--- Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- Columns & Data Types ---")
print(df.info())

--- Dataset Shape ---
Rows: 11430, Columns: 89

--- Columns & Data Types ---
<class 'pandas.DataFrame'>
RangeIndex: 11430 entries, 0 to 11429
Data columns (total 89 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   url                         11430 non-null  str    
 1   length_url                  11430 non-null  int64  
 2   length_hostname             11430 non-null  int64  
 3   ip                          11430 non-null  int64  
 4   nb_dots                     11430 non-null  int64  
 5   nb_hyphens                  11430 non-null  int64  
 6   nb_at                       11430 non-null  int64  
 7   nb_qm                       11430 non-null  int64  
 8   nb_and                      11430 non-null  int64  
 9   nb_or                       11430 non-null  int64  
 10  nb_eq                       11430 non-null  int64  
 11  nb_underscore               11430 non-null  int64  
 12  nb_tilde              

check for null values

In [57]:
df.isnull().sum().sum()

np.int64(0)

In [58]:
df['url'].isnull().value_counts()

url
False    11430
Name: count, dtype: int64

In [59]:
df['length_url'].isnull().value_counts()

length_url
False    11430
Name: count, dtype: int64

In [60]:
null_percentage = (df.isnull().sum() / len(df)) * 100
print(null_percentage[null_percentage > 0])

Series([], dtype: float64)


Identifying duplicates

In [61]:
df.duplicated().sum()

np.int64(0)

In [62]:
df.shape

(11430, 89)

handle duplicates

In [63]:
df = df.drop_duplicates()

In [64]:
df = df.dropna()

In [65]:
df.shape

(11430, 89)

In [66]:
# 2. Target එක 0 සහ 1 බවට හැරවීම
df['status'] = df['status'].map({
    'legitimate': 0,
    'phishing': 1
})

# Target values check කිරීම
print(df['status'].value_counts())

status
0    5715
1    5715
Name: count, dtype: int64


In [67]:
# 3. Features තෝරාගැනීම
features = [
    'length_url',
    'nb_dots',
    'nb_hyphens',
    'nb_slash',
    'nb_subdomains',
    'prefix_suffix',
    'shortening_service',
    'phish_hints',
    'ip',
    'https_token'
]

X = df[features]
y = df['status']

print("Selected Features:")
print(features)

print("\nX Shape:", X.shape)
print("y Shape:", y.shape)

Selected Features:
['length_url', 'nb_dots', 'nb_hyphens', 'nb_slash', 'nb_subdomains', 'prefix_suffix', 'shortening_service', 'phish_hints', 'ip', 'https_token']

X Shape: (11430, 10)
y Shape: (11430,)


In [68]:
from sklearn.model_selection import train_test_split

# 3. Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data shape: (9144, 10)
Testing data shape: (2286, 10)

Training target distribution:
status
0    4572
1    4572
Name: count, dtype: int64

Testing target distribution:
status
1    1143
0    1143
Name: count, dtype: int64


In [69]:
from sklearn.preprocessing import StandardScaler

# 4. Feature Scaling
scaler = StandardScaler()

# Training data   scaler  fit
X_train_scaled = scaler.fit_transform(X_train)

# Test data  scaler  apply
X_test_scaled = scaler.transform(X_test)

print("Original Training Shape:", X_train.shape)
print("Scaled Training Shape:", X_train_scaled.shape)

print("Original Testing Shape:", X_test.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)

Original Training Shape: (9144, 10)
Scaled Training Shape: (9144, 10)
Original Testing Shape: (2286, 10)
Scaled Testing Shape: (2286, 10)


In [70]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [71]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

# 1. Logistic Regression
lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr_model.fit(X_train_scaled, y_train)


# 2. Decision Tree
dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train_scaled, y_train)


# 3. Gradient Boosting
gb_model = GradientBoostingClassifier(
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)


print("All models trained successfully!")

All models trained successfully!


In [72]:
# Logistic Regression prediction
lr_pred = lr_model.predict(X_test_scaled)

# Decision Tree prediction
dt_pred = dt_model.predict(X_test_scaled)

# Gradient Boosting prediction
gb_pred = gb_model.predict(X_test_scaled)

print("All predictions completed successfully!")

All predictions completed successfully!


In [75]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = rf_model.predict(X_test_scaled)

models = {
    "Logistic Regression": (lr_model, lr_pred),
    "Decision Tree": (dt_model, dt_pred),
    "Random Forest": (rf_model, y_pred),
    "Gradient Boosting": (gb_model, gb_pred)
}

results = []

for model_name, (model, predictions) in models.items():

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

print("Model evaluation completed!")

Model evaluation completed!


In [76]:
import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model  Accuracy  Precision   Recall  F1-Score
Logistic Regression  0.760717   0.819742 0.668416  0.736386
      Decision Tree  0.782152   0.803959 0.746282  0.774047
      Random Forest  0.811024   0.814324 0.805774  0.810026
  Gradient Boosting  0.804899   0.851665 0.738408  0.791003


In [77]:
best_model_row = results_df.loc[
    results_df["F1-Score"].idxmax()
]

print("Best Model:")
print(best_model_row)

Best Model:
Model        Random Forest
Accuracy          0.811024
Precision         0.814324
Recall            0.805774
F1-Score          0.810026
Name: 2, dtype: object


In [78]:
import joblib

joblib.dump(rf_model, 'phishing_rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(features, 'features_list.pkl')

print("Final Random Forest model saved successfully!")

Final Random Forest model saved successfully!


In [79]:
loaded_model = joblib.load('phishing_rf_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
loaded_features = joblib.load('features_list.pkl')

print("Model loaded:", type(loaded_model).__name__)
print("Scaler loaded:", type(loaded_scaler).__name__)
print("Features loaded:", loaded_features)

Model loaded: RandomForestClassifier
Scaler loaded: StandardScaler
Features loaded: ['length_url', 'nb_dots', 'nb_hyphens', 'nb_slash', 'nb_subdomains', 'prefix_suffix', 'shortening_service', 'phish_hints', 'ip', 'https_token']
